In [5]:
import kagglehub
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
import tensorflow as tf
import numpy as np
from sklearn.metrics import accuracy_score

# Download the chest X-ray image dataset from Kaggle
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
# print("Path to dataset files:", path)

# Navigate into the chest_xray subfolder which contains train/val/test splits
dataset_path = Path(path) / "chest_xray"
# print(dataset_path)

# Define paths for the merged and final split datasets
merged_dir = Path("merged_dataset")
output_dir = Path("dataset_70_30")

# Define class names and original dataset splits to merge
classes = ["NORMAL", "PNEUMONIA"]
splits = ["train", "val", "test"]

# Clean old folders to avoid mixing data from previous runs
if merged_dir.exists():
    shutil.rmtree(merged_dir)
if output_dir.exists():
    shutil.rmtree(output_dir)

# Create merged folder structure with one subfolder per class
for c in classes:
    (merged_dir / c).mkdir(parents=True, exist_ok=True)

# Merge all images from train/val/test splits into a single folder per class
# Images are renamed with their split prefix to avoid filename conflicts
counter = 0
for split in splits:
    for c in classes:
        source = dataset_path / split / c
        if not source.exists():
            continue
        for img in source.iterdir():
            if img.suffix.lower() not in [".jpeg", ".jpg", ".png"]:
                continue
            new_name = f"{split}_{counter}_{img.name}"
            shutil.copy(img, merged_dir / c / new_name)
            counter += 1
# print(f"Merging completed. Total images copied: {counter}")

# Define train and test output directories for the 70/30 split
train_dir = output_dir / "train"
test_dir = output_dir / "test"

# Collect all merged image paths and their corresponding class labels
all_images = []
all_labels = []

for c in classes:
    class_folder = merged_dir / c
    for img in class_folder.iterdir():
        if img.suffix.lower() in [".jpeg", ".jpg", ".png"]:
            all_images.append(img)
            all_labels.append(c)
# print("Total merged images:", len(all_images))

# Split into 70% training and 30% test datasets, stratified to keep class balance equal in both datasets
train_imgs, test_imgs, train_labels, test_labels = train_test_split(
    all_images,
    all_labels,
    test_size=0.30,
    random_state=42,
    stratify=all_labels
)
# print("Train images:", len(train_imgs))
# print("Test images:", len(test_imgs))

# Create training and test folder structure with one subfolder per class
for c in classes:
    (train_dir / c).mkdir(parents=True, exist_ok=True)
    (test_dir / c).mkdir(parents=True, exist_ok=True)

# Copy training images into the train directory
for img, label in zip(train_imgs, train_labels):
    shutil.copy(img, train_dir / label / img.name)

# Copy test images into the test directory
for img, label in zip(test_imgs, test_labels):
    shutil.copy(img, test_dir / label / img.name)

# print("70/30 split created successfully")
# print("Training data saved in:", train_dir)
# print("Test data saved in:", test_dir)

# Load train and test chest X-ray images as grayscale batches and normalize pixel values to [0, 1]
train_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset_70_30/train", image_size=(180, 180), batch_size=32, color_mode="grayscale", shuffle=True, seed=42
).map(lambda x, y: (x / 255.0, y))

test_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset_70_30/test", image_size=(180, 180), batch_size=32, color_mode="grayscale", shuffle=False
).map(lambda x, y: (x / 255.0, y))

# Create, compile and train the model on the training dataset
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(180, 180, 1)),
    tf.keras.layers.Conv2D(100, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(100, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(100, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Stop training early if val_accuracy does not improve for 5 epochs and restore best weights
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True
)

model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=20,
    callbacks=[early_stop]
)

# Non-shuffled train dataset for consistent evaluation
train_ds_eval = tf.keras.utils.image_dataset_from_directory(
    "dataset_70_30/train", image_size=(180, 180), batch_size=32,
    color_mode="grayscale", shuffle=False
).map(lambda x, y: (x / 255.0, y))

# Extract true labels from all batches
y_train = np.concatenate([y.numpy() for x, y in train_ds_eval])
y_test = np.concatenate([y.numpy() for x, y in test_ds])

# Generate predictions for training and test datasets
nn_model_prediction_train = (model.predict(train_ds_eval) > 0.5).astype(int).flatten()
nn_model_prediction_test = (model.predict(test_ds) > 0.5).astype(int).flatten()

# Evaluate and display model accuracy with Accuracy score for training and test datasets
print(f"\nAccuracy Score for training dataset using Convolutional Neural Network model: {accuracy_score(y_train, nn_model_prediction_train):.3f}")
print(f"Accuracy Score for test dataset using Convolutional Neural Network model: {accuracy_score(y_test, nn_model_prediction_test):.3f}\n")

# Cleanup merged, split and originally downloaded datasets to free up disk space
if merged_dir.exists():
    shutil.rmtree(merged_dir)
if output_dir.exists():
    shutil.rmtree(output_dir)

kaggle_cache = Path.home() / ".cache" / "kagglehub" / "datasets" / "paultimothymooney"
if kaggle_cache.exists():
    shutil.rmtree(kaggle_cache)

100%|██████████| 2.29G/2.29G [01:10<00:00, 34.9MB/s]

Extracting files...


Found 4099 files belonging to 2 classes.
Found 1757 files belonging to 2 classes.
Epoch 1/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 44s 335ms/step - accuracy: 0.8495 - loss: 0.3525 - val_accuracy: 0.9038 - val_loss: 0.2384
Epoch 2/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 43s 331ms/step - accuracy: 0.9295 - loss: 0.1857 - val_accuracy: 0.9402 - val_loss: 0.1491
Epoch 3/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 42s 327ms/step - accuracy: 0.9385 - loss: 0.1606 - val_accuracy: 0.9482 - val_loss: 0.1385
Epoch 4/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 42s 326ms/step - accuracy: 0.9541 - loss: 0.1199 - val_accuracy: 0.9254 - val_loss: 0.2097
Epoch 5/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 42s 327ms/step - accuracy: 0.9641 - loss: 0.1007 - val_accuracy: 0.9562 - val_loss: 0.1260
Epoch 6/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 42s 327ms/step - accuracy: 0.9700 - loss: 0.0830 - val_accuracy: 0.9545 - val_loss: 0.1274
Epoch 7/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 43s 329ms/step - accuracy: 0.9751 - loss: 0.0722 - val_accuracy: 0.9573 - val_loss: 0.1198
E

2026-04-14 16:39:20.965118: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


129/129 ━━━━━━━━━━━━━━━━━━━━ 10s 77ms/step
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 77ms/step

Accuracy Score for training dataset using Convolutional Neural Network model: 0.998
Accuracy Score for test dataset using Convolutional Neural Network model: 0.962

